In [ ]:
import os, sys

print("ISSM_DIR =", os.getenv("ISSM_DIR"))
issm_dir = os.getenv("ISSM_DIR")
sys.path.insert(0, issm_dir + "/bin")
sys.path.insert(0, issm_dir + "/lib")
sys.path.insert(0, issm_dir + "/share")
sys.path.insert(0, issm_dir + "/share/proj")

custom = "/home/yanmeiti/issm/Functions"
sys.path.insert(0, custom)
import generic
print("generic from:", generic.__file__)

sys.path.append(os.getenv('ISSM_DIR') + '/bin')
sys.path.append(os.getenv('ISSM_DIR') + '/lib')
sys.path.append(os.getenv('ISSM_DIR') + '/share')
sys.path.append(os.getenv('ISSM_DIR') + '/share/proj')

import numpy as np
from netCDF4 import Dataset
from scipy.interpolate import griddata
from model import model
from triangle import triangle
from InterpFromGridToMesh import InterpFromGridToMesh
from setmask import setmask
from model import *
from m1qn3inversion import m1qn3inversion
from generic import generic
from solve import solve
from export_netCDF import export_netCDF
from verbose import verbose
from setflowequation import setflowequation
import numpy as np
from triangle import triangle
from model import *
from netCDF4 import Dataset
from InterpFromGridToMesh import InterpFromGridToMesh
from bamg import bamg
from xy2ll import xy2ll
from plotmodel import plotmodel
from loadmodel import loadmodel
from setmask import setmask
from parameterize import parameterize
from socket import gethostname
from ll2xy import ll2xy
from BamgTriangulate import BamgTriangulate
from scipy.interpolate import griddata
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from ContourToMesh import ContourToMesh
import rasterio
from InterpFromMeshToMesh2d import InterpFromMeshToMesh2d
from paterson import paterson
import matplotlib.pyplot as plt
from averaging import averaging 
from slope import slope

print('done')




In [ ]:
def InterpFromMeshToIsmipGrid(index, xmesh, ymesh, data, resolution, default_value):
    import numpy as np
    from InterpFromMeshToGrid import InterpFromMeshToGrid
    
    # Define ismip6 grid
    r1=1000
    r2=resolution
    resolution_ismip6 = r1*r2
    
    # the resolution of ISMIP6-Greenland is 5km
    x_ismip6                =   r1*np.arange(-720,960,r2)
    y_ismip6                =   r1*np.arange(-3450,-570,r2)
    x_spacing_ismip6        =   r1*r2
    y_spacing_ismip6        =   r1*r2
    
    [X_ismip6, Y_ismip6] = np.meshgrid(x_ismip6, y_ismip6);

    if data.ndim > 1:
        data_interp = np.zeros((X_ismip6.shape[0],X_ismip6.shape[1],data.shape[0]))
        for i in range(0,data.shape[0]):
            data_interp[:,:,i] = np.flipud(InterpFromMeshToGrid(index, xmesh, ymesh, data[i,:], x_ismip6, y_ismip6, default_value))
    else:
        data_interp = np.flipud(InterpFromMeshToGrid(index, xmesh, ymesh, data, x_ismip6, y_ismip6, default_value))

    return data_interp, x_ismip6, y_ismip6

## Set Domain

In [ ]:
steps = [1]

if 1 in steps:
    print('   Step 1: Domain')

    #Generate initial uniform mesh (resolution = 20000 m)
    md = triangle(model(), '/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/Data/outerDomain.exp', 500)
    plotmodel(md, 'data', 'mesh')
    print('   Step 1: Mesh creation')
    
    #### Option 4: average MEASURES velocity map
    ncdata = Dataset('/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/Data/icevel_measures_mean.nc', mode='r')
    x1 = np.squeeze(ncdata.variables['x'][:].data)
    y1 = np.squeeze(ncdata.variables['y'][:].data)
    velx = np.squeeze(ncdata.variables['velx'][:].data)
    vely = np.squeeze(ncdata.variables['vely'][:].data)
    ncdata.close()

    velocity = (np.sqrt(velx**2 + vely**2))
    
    vel = InterpFromGridToMesh(x1, y1, np.flipud(velocity.T), md.mesh.x, md.mesh.y, 0)
    
    md = bamg(md, 'hmax', 40000, 'hmin', 500, 'gradation', 1.4, 'field', vel, 'err', 8)


    plotmodel(md, 'data', 'mesh')
    export_netCDF(md, '/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/Models_tym_v2/Step1.1_1850mesh.nc')
    

In [ ]:
# # print('   Loading 1850 Surface')
# ncdata = Dataset('../Data/elev1850_MD_mean.nc', mode='r')
# X1850 = ncdata.variables['X_ismip6'][:].data
# Y1850 = ncdata.variables['Y_ismip6'][:].data
# isur1850 = ncdata.variables['elev1850_mean_EIGEN6C4'][:].data
# ncdata.close()

# x1850 = X1850[:,0]
# y1850 = Y1850[0,:]
# print(y1850.shape)
# print(x1850.shape)
# print(isur1850.shape)

# md.geometry.surface = InterpFromGridToMesh(x1850, y1850, np.flipud(isur1850.T), md.mesh.x, md.mesh.y, 0)
# plotmodel(md, 'data', md.geometry.surface)
# # np.nansum(Y1850[0,:]-Y1850[1,:])

## Parameterization

In [ ]:
steps = [2]

if 2 in steps:
    loadname = '/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/Models_tym_v2/Step1.1_1850mesh.nc'
    md = loadmodel(loadname)

    md.mask.ice_levelset=np.ones(md.mesh.numberofvertices)
    inn=ContourToMesh(md.mesh.elements,md.mesh.x,md.mesh.y,'/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/Data/innerDomainLIA.exp','node',1)
    md.mask.ice_levelset[np.where(inn==1)]=-1

    
    md = parameterize(md, '/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/scripts_tym_v2/S0_par.py')
    md = setflowequation(md, 'SSA', 'all')

    
    plotmodel(md, 'data',inn)
    plotmodel(md, 'data',md.mask.ice_levelset)
    export_netCDF(md, '/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/Models_tym_v2/Step1.2_parameters.nc')

In [ ]:
## check the new floating area
import numpy as np
from plotmodel import plotmodel

print('ice<0 count =', np.sum(md.mask.ice_levelset < 0))
print('ocean<0 count =', np.sum(md.mask.ocean_levelset < 0))
print('floating count =', np.sum((md.mask.ice_levelset < 0) & (md.mask.ocean_levelset < 0)))
print('grounded count =', np.sum((md.mask.ice_levelset < 0) & (md.mask.ocean_levelset > 0)))

print('unique ocean_levelset:', np.unique(md.mask.ocean_levelset[:1000]))
print('ocean min/max:', md.mask.ocean_levelset.min(), md.mask.ocean_levelset.max())


floating_mask = ((md.mask.ice_levelset < 0) & (md.mask.ocean_levelset < 0)).astype(float)
grounded_mask = ((md.mask.ice_levelset < 0) & (md.mask.ocean_levelset > 0)).astype(float)

plotmodel(md, 'data', floating_mask, 'title', 'Floating ice')
# plotmodel(md, 'data', grounded_mask, 'title', 'Grounded ice')


In [ ]:
steps = [3]

if 3 in steps:
    print('   Step 3: Solve Stress Balance Solution to check velocity')
    loadname = '/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/Models_tym_v2/Step1.2_parameters.nc'
    md = loadmodel(loadname)
    
    
    #Control general
    md.inversion.iscontrol = 0

    #Additional parameters
    md.stressbalance.restol = 0.01
    md.stressbalance.reltol = 0.1
    md.stressbalance.abstol = np.nan
    md.stressbalance.loadingforce = np.zeros((md.mesh.numberofvertices, 3)) ###

    #Go solve
    md.cluster = generic('name', gethostname(), 'np', 8)
    md.toolkits = toolkits()
    md.verbose = verbose('solution', True, 'control', True)
    md = solve(md, 'Stressbalance')
    

    savename = '/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/Models_tym_v2/Step1.3_stressbalance.nc'
    export_netCDF(md, savename)
    # print("\n !!! Warning: SIA's model is not consistent on ice shelves !!!\n")

    

In [ ]:


[velobs_ismip, x_ismip6, y_ismip6] =  InterpFromMeshToIsmipGrid(md.mesh.elements, md.mesh.x, md.mesh.y, md.initialization.vel, 1, 0)
[velstress_ismip, x_ismip6, y_ismip6] =  InterpFromMeshToIsmipGrid(md.mesh.elements, md.mesh.x, md.mesh.y, np.squeeze(md.results.StressbalanceSolution.Vel), 1, 0)
veldif = velstress_ismip - velobs_ismip

# Mask zeros
velobs_masked = np.ma.masked_where(velobs_ismip == 0, velobs_ismip)
velstress_masked = np.ma.masked_where(velstress_ismip == 0, velstress_ismip)
veldif_masked = np.ma.masked_where(veldif == 0, veldif)

# Create Figure
fig, axs = plt.subplots(1, 3, figsize=(12, 4))

cmap = plt.cm.viridis.copy()
cmap.set_bad(color='none') 

# Painel 1
im1 = axs[0].imshow(np.flipud(velobs_masked), origin='lower', cmap=cmap)
im1.set_clim(0, 100)
axs[0].set_title('Observed')
cbar1 = fig.colorbar(im1, ax=axs[0], fraction=0.046, pad=0.04)
cbar1.set_label('Velocity (m/yr)')

# Panel 2
im2 = axs[1].imshow(np.flipud(velstress_masked), origin='lower', cmap=cmap)
im2.set_clim(0, 100)
axs[1].set_title('Modeled')
cbar2 = fig.colorbar(im2, ax=axs[1], fraction=0.046, pad=0.04)
cbar2.set_label('Velocity (m/yr)')

# Panel 3
im3 = axs[2].imshow(np.flipud(veldif_masked), origin='lower', cmap=cmap)
im3.set_clim(-10, 10)
axs[2].set_title('Difference')
cbar3 = fig.colorbar(im3, ax=axs[2], fraction=0.046, pad=0.04)
cbar3.set_label('Difference (m/yr)')

plt.tight_layout()
# plt.savefig('/Users/amoraesl/Documents/AGU2025/ISSM/velocities_comparison_inversion.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
steps = [4]

if 4 in steps:
    #   Step 3: Control method friction {{{
    print('   Step 3: Control method friction')
    md = loadmodel('/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/Models_tym_v2/Step1.3_stressbalance.nc')
    #Control general
    md.inversion.iscontrol = 1 
    md.inversion.nsteps = 100
    md.inversion.step_threshold = 0.99 * np.ones((md.inversion.nsteps))
    md.inversion.maxiter_per_step = 5 * np.ones((md.inversion.nsteps))

    #Cost functions
    md.inversion.cost_functions = [101, 103, 501]
    md.inversion.cost_functions_coefficients = np.ones((md.mesh.numberofvertices, 3))
    md.inversion.cost_functions_coefficients[:, 0] = 350
    md.inversion.cost_functions_coefficients[:, 1] = 0.2
    md.inversion.cost_functions_coefficients[:, 2] = 2e-6

    #Controls
    md.inversion.control_parameters = ['FrictionCoefficient']
    md.inversion.gradient_scaling = 50 * np.ones((md.inversion.nsteps, 1))
    md.inversion.min_parameters = 1 * np.ones((md.mesh.numberofvertices, 1))
    md.inversion.max_parameters = 200 * np.ones((md.mesh.numberofvertices, 1))

    # Fix: lock floating ice friction to 0 during inversion
    floating_nodes = (md.mask.ice_levelset < 0) & (md.mask.ocean_levelset < 0)
    md.inversion.min_parameters[floating_nodes, 0] = 0
    md.inversion.max_parameters[floating_nodes, 0] = 0

    #Additional parameters
    md.stressbalance.restol = 0.01
    md.stressbalance.reltol = 0.1
    md.stressbalance.abstol = np.nan
    md.stressbalance.loadingforce = np.zeros((md.mesh.numberofvertices, 3))

    #Go solve
    md.cluster = generic('name', gethostname(), 'np', 4)
    md.toolkits = toolkits()
    md.verbose = verbose('solution', True, 'control', True)
    md = solve(md, 'Stressbalance')


    #Update model friction fields accordingly
    friction_guess = md.friction.coefficient
    md.friction.coefficient = md.results.StressbalanceSolution.FrictionCoefficient
    # Fix: explicitly zero friction on floating ice after inversion
    floating_nodes = (md.mask.ice_levelset < 0) & (md.mask.ocean_levelset < 0)
    md.friction.coefficient[floating_nodes] = 0

    savename = '/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/Models_tym_v2/Step1.4_BasalInversion.nc'

    export_netCDF(md, savename)

In [ ]:


[velobs_ismip, x_ismip6, y_ismip6] =  InterpFromMeshToIsmipGrid(md.mesh.elements, md.mesh.x, md.mesh.y, md.initialization.vel, 1, 0)
[velstress_ismip, x_ismip6, y_ismip6] =  InterpFromMeshToIsmipGrid(md.mesh.elements, md.mesh.x, md.mesh.y, np.squeeze(md.results.StressbalanceSolution.Vel), 1, 0)
veldif = velstress_ismip - velobs_ismip

# Mask zeros
velobs_masked = np.ma.masked_where(velobs_ismip == 0, velobs_ismip)
velstress_masked = np.ma.masked_where(velstress_ismip == 0, velstress_ismip)
veldif_masked = np.ma.masked_where(veldif == 0, veldif)

# Create Figure
fig, axs = plt.subplots(1, 3, figsize=(12, 4))

cmap = plt.cm.viridis.copy()
cmap.set_bad(color='none') 

# Painel 1
im1 = axs[0].imshow(np.flipud(velobs_masked), origin='lower', cmap=cmap)
im1.set_clim(0, 100)
axs[0].set_title('Observed')
cbar1 = fig.colorbar(im1, ax=axs[0], fraction=0.046, pad=0.04)
cbar1.set_label('Velocity (m/yr)')

# Panel 2
im2 = axs[1].imshow(np.flipud(velstress_masked), origin='lower', cmap=cmap)
im2.set_clim(0, 100)
axs[1].set_title('Modeled')
cbar2 = fig.colorbar(im2, ax=axs[1], fraction=0.046, pad=0.04)
cbar2.set_label('Velocity (m/yr)')

# Panel 3
im3 = axs[2].imshow(np.flipud(veldif_masked), origin='lower', cmap=cmap)
im3.set_clim(-10, 10)
axs[2].set_title('Difference')
cbar3 = fig.colorbar(im3, ax=axs[2], fraction=0.046, pad=0.04)
cbar3.set_label('Difference (m/yr)')

plt.tight_layout()
# plt.savefig('/Users/amoraesl/Documents/AGU2025/ISSM/velocities_comparison_inversion.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
steps = [5]

if 5 in steps:
    print('   Step 5: Solve Stress Balance Solution (Post-Inversion)')
    loadname = '/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/Models_tym_v2/Step1.4_BasalInversion.nc'
    md = loadmodel(loadname)
    
    
    #Control general
    md.inversion.iscontrol = 0

    #Additional parameters
    md.stressbalance.restol = 0.01
    md.stressbalance.reltol = 0.1
    md.stressbalance.abstol = np.nan
    md.stressbalance.loadingforce = np.zeros((md.mesh.numberofvertices, 3)) ###

    #Go solve
    md.cluster = generic('name', gethostname(), 'np', 8)
    md.toolkits = toolkits()
    md.verbose = verbose('solution', True, 'control', True)
    md = solve(md, 'Stressbalance')


    
    savename = '/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/Models_tym_v2/Step1.5_StressBalance_PostInversion.nc'
    export_netCDF(md, savename)

    

In [ ]:
# check friction on the floating area after inversion
loadname = '/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/Models_tym_v2/Step1.5_StressBalance_PostInversion.nc'
md = loadmodel(loadname)

plotmodel(md, 'data', md.friction.coefficient,
          'title', 'Friction Coefficient (Post-Inversion)',
          'xlim', [-250000, -150000],
          'ylim', [-2350000, -2250000])


In [ ]:
# Run this in cell[13] or a new cell
loadname = '/home/yanmeiti/issm/Greenland/ISSM_GrIS_Yanmei/Models_tym_v2/Step1.4_BasalInversion.nc'
md = loadmodel(loadname)

floating = (md.mask.ice_levelset < 0) & (md.mask.ocean_levelset < 0)
print(f'floating nodes: {int(floating.sum())}')
print(f'friction on floating: min={md.friction.coefficient[floating].min():.2f}, max={md.friction.coefficient[floating].max():.2f}')

# Plot floating ice area
plotmodel(md, 'data', floating.astype(float),
          'title', 'Floating ice mask',
          'xlim', [-250000, -150000],
          'ylim', [-2350000, -2250000])


plotmodel(md, 'data', md.friction.coefficient,
          'title', 'Friction Coefficient (Post-Inversion)',
          'xlim', [-250000, -150000],
          'ylim', [-2350000, -2250000])

In [ ]:
print("\n--- Checking Initial State (Post-Inversion) ---")
vel = md.initialization.vel
thickness = md.geometry.thickness
friction = md.friction.coefficient

print(f"Initial Velocity (m/yr) - Min: {np.min(vel):.2f}, Max: {np.max(vel):.2f}, Mean: {np.mean(vel):.2f}")
print(f"Initial Thickness (m)   - Min: {np.min(thickness):.2f}, Max: {np.max(thickness):.2f}")
print(f"Friction Coefficient    - Min: {np.min(friction):.2f}, Max: {np.max(friction):.2f}")
print(f"StressbalanceSolution.Vel - Min: {np.min(md.results.StressbalanceSolution.Vel):.2f}, Max: {np.max(md.results.StressbalanceSolution.Vel):.2f}")


In [ ]:
ice = md.mask.ice_levelset < 0
floating = ice & (md.mask.ocean_levelset < 0)
grounded = ice & (md.mask.ocean_levelset > 0)
noice = md.mask.ice_levelset > 0

print('all bed>base        =', np.sum(md.geometry.bed > md.geometry.base))
print('ice bed>base        =', np.sum(ice & (md.geometry.bed > md.geometry.base)))
print('grounded bed!=base  =', np.sum(grounded & (np.abs(md.geometry.bed - md.geometry.base) > 1e-6)))
print('floating bed>=base  =', np.sum(floating & (md.geometry.bed >= md.geometry.base)))
print('ocean bed>base      =', np.sum(noice & (md.geometry.bed > md.geometry.base)))
